# The full workflow in Python

The Studio's workflow done with the core API: pick a network, place participants,
set up the market, check it with the bidding rule, define an experiment, start it on
the cluster and copy its results back. From there the
[Studio](../../../studio/app.py)'s **Results** page shows them, so this notebook
stops where the Studio's inspection begins.

Each section names the Studio page it stands for. Everything here is plain
`core`: no Studio code is imported. Nothing is saved to `config/`. The scenario is
written next to this notebook, and the experiment carries its scenario inside it.

| Step | Studio | Core API |
|---|---|---|
| 1. Network | Inspect | [`core.grid.simbench`](../../../core/grid/simbench.py) |
| 2. Scenario | Scenario | [`core.config.scenario`](../../../core/config/scenario.py), [`placement`](../../../core/config/placement.py), [`store`](../../../core/config/store.py) |
| 3. One day of the market | Validate → Market → Step | [`core.market.lem.LocalEnergyMarket`](../../../core/market/lem.py) |
| 4. Baseline and stress | Validate → Market → Baseline | [`core.market.baseline`](../../../core/market/baseline.py) |
| 5. The RL environment | (Summary) | [`core.env.MarketEnv`](../../../core/env/market_env.py) |
| 6. Experiment | Experiments | [`core.runner.Experiment`](../../../core/runner/spec.py) |
| 7. Start, watch, sync | Runs | [`core.runner.targets`](../../../core/runner/targets.py) |
| 8. First look at results | Results | [`core.runner.results`](../../../core/runner/results.py) |

Run it in the `marlem` environment. Sections 1 to 6 run on this PC in about a
minute. Section 7 talks to the cluster and starts only when `START = True`.

In [1]:
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "market").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

HERE = ROOT / "notebooks" / "market" / "demo"

## 1. Pick a network *(Studio: Inspect)*

SimBench's networks by voltage level and scenario: `--0-` is today, `--1-` and
`--2-` are future scenarios with more PV and storage. `summary` loads one network
(cached on disk after the first time). `kind_counts` shows what its buses carry,
which is what participants are placed by.

The demo uses `1-LV-rural1--2-sw`: 13 buses with elements, all with a load, 8 with
PV and 5 with a SimBench storage unit. That is small enough to play quickly and has
everything the market needs.

In [2]:
from core.config import placement
from core.config.scenario import BatterySetup, DsoSetup, GridSetup, MarketSetup, Scenario
from core.grid import simbench
from core.grid.contract import NetworkRef

lv = [n for n in simbench.list_networks("simbench", level="LV") if n.scenario == 2]
display(pd.DataFrame([{"code": n.ref.code, "label": n.label, "switches": n.switches} for n in lv]))

CODE = "1-LV-rural1--2-sw"
ref = NetworkRef("simbench", CODE)
info = simbench.summary(ref)
print(f"{CODE}: {info.lines} lines, {info.transformers} transformer(s), "
      f"load {info.total_load_kw:g} kW, generation {info.total_generation_kw:g} kW (rated)")

,code,label,switches
0,1-LV-rural1--2-sw,LV rural 1 · scenario 2 · with switches,True
1,1-LV-rural1--2-no_sw,LV rural 1 · scenario 2 · no switches,False
2,1-LV-rural2--2-sw,LV rural 2 · scenario 2 · with switches,True
3,1-LV-rural2--2-no_sw,LV rural 2 · scenario 2 · no switches,False
4,1-LV-rural3--2-sw,LV rural 3 · scenario 2 · with switches,True
5,1-LV-rural3--2-no_sw,LV rural 3 · scenario 2 · no switches,False
6,1-LV-semiurb4--2-sw,LV semi-urban 4 · scenario 2 · with switches,True
7,1-LV-semiurb4--2-no_sw,LV semi-urban 4 · scenario 2 · no switches,False
8,1-LV-semiurb5--2-sw,LV semi-urban 5 · scenario 2 · with switches,True
9,1-LV-semiurb5--2-no_sw,LV semi-urban 5 · scenario 2 · no switches,False


1-LV-rural1--2-sw: 13 lines, 1 transformer(s), load 181.3 kW, generation 468.2 kW (rated)

## 2. Build the scenario *(Studio: Scenario)*

A [`Scenario`](../../../core/config/scenario.py) is plain data in four parts:

- `grid`: the network, the day, the limits and the scaling factors;
- `participants`: who sits on which bus and owns which elements;
- `market`: the clearing rules and the bidding rule's settings;
- `dso`: tariffs, fees and who pays the losses.

The Studio's Scenario page edits exactly these fields.

In [3]:
scenario = Scenario(
    name="demo_rural1",
    grid=GridSetup(code=CODE, day="2016-06-15"),
    market=MarketSetup(partner_preference=True),
    dso=DsoSetup(prices="de_2024", voltage_drop="measured", losses="bcdla"),
)
print(placement.kind_counts(scenario))

{'load+pv': 8, 'load+storage': 5}

**Placing participants.** `place_all` puts one participant on every bus of the
given kinds. Each participant takes over the bus's loads, PV and storage. A
SimBench storage unit becomes the participant's battery, with its own capacity and
power limit. `toggle(scenario, bus)` adds or removes a single one, which is what
clicking a bus in the Studio does.

`set_battery` gives a participant a battery of your own. `battery_preset` sizes one
like SimBench's units (power = capacity ÷ 2 h). Here one PV household gets a
typical battery.

In [4]:
placement.place_all(scenario, ["load+pv", "load+storage"])

pv_only = next(p for p in scenario.participants if p.battery is None)
print(placement.set_battery(scenario, pv_only.id, placement.battery_preset("typical")))

kinds = simbench.element_kinds(ref)
display(pd.DataFrame([
    {
        "id": p.id,
        "bus": p.bus,
        "owns": ", ".join(sorted({kinds[e] for e in p.owns})),
        "battery kWh": p.battery.capacity_kwh if p.battery else None,
        "power kW": p.battery.max_power_kw if p.battery else None,
    }
    for p in scenario.participants
]))

P1: battery saved

,id,bus,owns,battery kWh,power kW
0,P1,LV1.101 Bus 1,"load, sgen",13.7,6.85
1,P2,LV1.101 Bus 10,"load, storage",100.5,50.20
2,P3,LV1.101 Bus 11,"load, sgen",NaN,NaN
3,P4,LV1.101 Bus 12,"load, storage",146.7,73.40
4,P5,LV1.101 Bus 13,"load, sgen",NaN,NaN
5,P6,LV1.101 Bus 14,"load, storage",61.1,30.60
6,P7,LV1.101 Bus 2,"load, sgen",NaN,NaN
7,P8,LV1.101 Bus 3,"load, sgen",NaN,NaN
8,P9,LV1.101 Bus 5,"load, sgen",NaN,NaN
9,P10,LV1.101 Bus 6,"load, storage",36.7,18.30


**Checking and saving.** `problems` is the list the Studio shows in red. It must be
empty before anything plays. `store.save` writes the JSON the Studio reads, so the
scenario can also be opened on the Scenario page.

In [5]:
problems = scenario.problems(kinds)
print(problems or "no problems")

from core.config import store

SCENARIO_FILE = HERE / "demo_scenario.json"
store.save(scenario, SCENARIO_FILE)
print("saved", SCENARIO_FILE.relative_to(ROOT))

no problems

saved

notebooks\market\demo\demo_scenario.json

## 3. One day of the market *(Studio: Validate → Market → Step)*

[`LocalEnergyMarket`](../../../core/market/lem.py) plays the scenario. `reset`
loads the day's profiles. `step` plays one step (orders from the bidding rule,
clearing, the power flow, fees, settlement, reputation). `run_to_end` plays the
rest of the day. Each step returns a `StepRecord`: what the Studio's Step page
shows, phase by phase.

In [6]:
from core.market.lem import LocalEnergyMarket, net_bill

market = LocalEnergyMarket(scenario)
market.reset()
records = market.run_to_end()

day = pd.DataFrame([
    {
        "hour": r.step * scenario.grid.step_hours,
        "orders": len(r.orders),
        "p2p kWh": r.p2p_kwh,
        "dso kWh": r.dso_kwh,
        "price": r.clearing_price,
        "fees": r.fees,
        "line max %": r.grid.max_line_loading_percent,
        "team money": sum(net_bill(r, pid) for pid in r.participants),
    }
    for r in records
]).set_index("hour")
display(day.loc[8:16])
print(f"day: {day['p2p kWh'].sum():.1f} kWh between peers, team money {day['team money'].sum():.2f}")

,orders,p2p kWh,dso kWh,price,fees,line max %,team money
hour,,,,,,,
8.0,8,0.000,95.886,0.245,0.038,19.45,9.184
9.0,8,0.000,139.793,0.245,0.486,27.03,9.873
10.0,8,0.000,149.842,0.245,0.534,26.93,8.893
11.0,8,0.000,154.116,0.245,0.496,25.81,7.847
12.0,8,0.000,133.221,0.245,0.154,23.32,6.093
13.0,8,1.672,79.378,0.180,0.009,13.98,2.394
14.0,9,0.207,65.364,0.185,0.002,12.69,0.872
15.0,9,0.915,47.662,0.180,0.001,9.94,0.681
16.0,9,1.837,32.718,0.202,0.002,7.39,1.599


day: 9.3 kWh between peers, team money -2.70

In [7]:
# One step in detail: noon's trades
noon = records[12]
display(pd.DataFrame([
    {"stage": s.stage, "seller": t.seller_id, "buyer": t.buyer_id, "kWh": t.quantity, "price": t.price, "fees": t.fees}
    for s in noon.stages for t in s.trades
]))

,stage,seller,buyer,kWh,price,fees
0,3,P1,DSO,34.363,0.054,1.039e-02
1,3,P3,DSO,25.981,0.054,5.977e-02
2,3,P5,DSO,7.104,0.054,3.069e-04
3,3,P7,DSO,5.872,0.054,6.342e-05
4,3,P8,DSO,14.387,0.054,3.418e-02
5,3,P9,DSO,28.455,0.054,2.985e-02
6,3,P11,DSO,12.162,0.054,8.291e-03
7,3,P12,DSO,4.897,0.054,1.103e-02


## 4. Baseline and stress *(Studio: Validate → Market → Baseline)*

[`baseline.play`](../../../core/market/baseline.py) plays a window of days with the
bidding rule without its noise (the evaluation's `net_position` baseline), carrying
the batteries from day to day. `summary` gives what the Baseline tab's overview row
shows.

`baseline.sweep` plays every combination of some settings' values: here
`grid.generation_scale` (MP-4i), to see where the network starts to violate its
limits. Unlike the Studio tab, a notebook has no 31-day limit, so a whole season or
year is possible (it takes longer: about a second per day on this network).

In [8]:
from core.market import baseline

WEEK = [str(d.date()) for d in pd.date_range("2016-07-01", "2016-07-07")]
week = baseline.play(scenario, WEEK)
display(pd.Series(week.summary()).to_frame("rule, ×1"))

,"rule, ×1"
steps,168.000
line_max_percent,41.330
line_p95_percent,30.268
trafo_max_percent,132.310
voltage_min_pu,1.019
voltage_max_pu,1.062
steps_violated,9.000
steps_voltage_low,0.000
steps_voltage_high,8.000
steps_line_overload,0.000


In [9]:
settings = ["grid.generation_scale"]
runs = baseline.sweep(scenario, WEEK, {settings[0]: [1, 2, 3, 4]})
table = baseline.overview(runs, settings)
display(table[settings + ["line_max_percent", "trafo_max_percent", "voltage_max_pu",
                          "steps_violated", "grid_losses_kwh", "team_money"]])

,grid.generation_scale,line_max_percent,trafo_max_percent,voltage_max_pu,steps_violated,grid_losses_kwh,team_money
0,1,41.33,132.31,1.062,9,171.651,-432.132
1,2,82.94,271.71,1.097,36,484.834,-348.419
2,3,122.67,406.38,1.127,55,1009.062,-362.429
3,4,161.00,537.74,1.155,62,1730.247,-442.823


To use a factor, set it on the scenario and save again, as the tab's *Use in the
scenario* does: `scenario.grid.generation_scale = 3; store.save(scenario, SCENARIO_FILE)`.
The demo keeps ×1.

## 5. The RL environment *(what the Summary page lists)*

Training does not use `LocalEnergyMarket` directly. It uses
[`MarketEnv`](../../../core/env/market_env.py), an RLlib multi-agent environment
around it: one agent per participant, the paper's observation, the mixed action
(side, quantity, price, partner) and a reward. The experiment builds it from an
`env_config`; here it is built by hand to see what the agents see.

`contract()` is what `run.json` records: the observation names, the action's parts
and the scales. Playing one day with `net_position_policy` (the rule, read from the
observation) gives the reward the baseline earns.

In [10]:
from core.env import MarketEnv, net_position_policy

env = MarketEnv({"scenario": str(SCENARIO_FILE), "days": WEEK, "seed": 0,
                 "reward": "bill", "observation": "paper"})
contract = env.contract()
print(len(env.agents_id), "agents;", len(contract["observation"]), "observation signals per agent")
print("action parts:", list(contract["action"]["components"]))

obs, _ = env.reset(seed=0)
rule = net_position_policy(env)
total, done = 0.0, False
while not done:
    obs, rewards, terminated, truncated, _ = env.step(rule(obs))
    total += sum(rewards.values())
    done = terminated["__all__"] or truncated["__all__"]
print(f"one day with the rule: team reward {total:.3f}")

13

agents;

54

observation signals per agent

action parts:

['side', 'quantity', 'price', 'partner']

one day with the rule: team reward -0.903

## 6. The experiment *(Studio: Experiments)*

An experiment is the JSON of
[`config/experiments/`](../../../config/experiments/), here written as a dict. The
fields are documented at the top of [`spec.py`](../../../core/runner/spec.py).
`Experiment.from_dict` validates it exactly as the Studio's editor does.

- The scenario goes in **inline** (`asdict(scenario)`), so the cluster needs no
  scenario file in its checkout.
- `train_days` and `test_periods`: the held-out days, here one week per season,
  are taken out of the training days automatically.
- `variants` are the arms. Each overwrites part of the base, here whether the fees are charged.
- `expand()` lists every run: algorithms × paradigms × variants × seeds.

In [11]:
from core.runner import Experiment

spec = {
    "name": "demo_workflow",
    "description": "The full-workflow notebook's demo: fees charged against not charged.",
    "scenario": asdict(scenario),
    "train_days": {"from": "2016-01-01", "to": "2016-12-30"},
    "test_periods": {
        "winter": {"from": "2016-01-11", "to": "2016-01-17"},
        "spring": {"from": "2016-04-11", "to": "2016-04-17"},
        "summer": {"from": "2016-07-11", "to": "2016-07-17"},
        "autumn": {"from": "2016-10-10", "to": "2016-10-16"},
    },
    "env_config": {"reward": "bill", "observation": "paper"},
    "algorithms": ["ppo"],
    "paradigms": ["ctde"],
    "seeds": [0, 1],
    "variants": {
        "fees": {"scenario": {"dso": {"fees_charged": True}}},
        "no_fees": {"scenario": {"dso": {"fees_charged": False}}},
    },
    "budget": 240000,
    "eval_points": 4,
    "resources": {"cpus": 4, "mem_gb": 6, "minutes": 240},
}
experiment = Experiment.from_dict(spec)
print(f"{len(experiment.train_days)} training days, {len(experiment.test_days)} held-out days")
display(pd.DataFrame([
    {"run": r.name, "paradigm": r.paradigm, "variant": r.variant, "seed": r.seed, "budget": r.budget}
    for r in experiment.expand()
]))

337 training days, 28 held-out days

,run,paradigm,variant,seed,budget
0,ppo_ctde_s0_fees,ctde,fees,0,240000
1,ppo_ctde_s1_fees,ctde,fees,1,240000
2,ppo_ctde_s0_no_fees,ctde,no_fees,0,240000
3,ppo_ctde_s1_no_fees,ctde,no_fees,1,240000


**The test version.** `test_version()` is the smallest run of the same experiment:
one seed, one training and one test day, a budget of one training batch. It proves
the pipeline (start, train, evaluate, write results) in minutes. The Studio's *Test*
preset starts the same thing (`profile="test"`).

In [12]:
test = experiment.test_version()
print(test.name, "|", len(test.expand()), "runs |", test.train_days, "->", test.test_days,
      "| budget", test.budget)

demo_workflow--test

|

2

runs |

['2016-01-01']

->

['2016-01-11']

| budget

{'ppo': 100}

## 7. Start, watch and sync *(Studio: Experiments → Start, then Runs)*

A [target](../../../hpc/targets.toml) is where runs train: `local` (this PC, a
detached worker) or `taltech` (SLURM, driven over SSH from here). Both have the
same methods:

- `check(experiment)`: whether it can start: the code committed and the same as
  the cluster's checkout, the resources within a partition. The Studio's Start
  button stays disabled while this lists problems.
- `start_new(experiment, profile)`: queue and start, under a dated name if the name
  is taken; returns the name.
- `runs(name)`: one `RunState` per run (state, progress, last evaluation), the Runs
  page's table.
- `sync(name)`: copy the results to `runs/@taltech/<name>/` (not the checkpoints,
  which stay on the cluster).

`START = False` keeps this notebook from starting anything. Set it to `True` to run
the test version; change `PROFILE` to `"full"` for the real thing.

In [ ]:
from core.runner import targets

TARGET, PROFILE, START = "taltech", "test", False

target = targets.get(TARGET)
report = target.check(experiment, profile=PROFILE)
print("ready" if report["ok"] else "NOT ready")
for problem in report["problems"]:
    print(" -", problem)

In [ ]:
if START:
    NAME = target.start_new(experiment, PROFILE)
    print("started", NAME)
else:
    NAME = None
    print("not started (START = False)")

In [ ]:
# Run again while it trains: the Runs page's table
if NAME:
    display(pd.DataFrame([asdict(s) for s in target.runs(NAME)])
            [["name", "state", "slurm_state", "progress", "env_steps", "last_eval", "heldout", "error"]])

In [ ]:
# Once every run is completed: copy the results here
if NAME:
    folder = target.sync(NAME)
    print(f"{target.copied} files copied to {folder}")

## 8. A first look, then the Studio *(Studio: Results)*

[`results`](../../../core/runner/results.py) reads run folders the same way the
Results page does: `find_runs` locates them (this PC and every cluster's mirror),
`load_runs` reads their settings and state, and `overview` gives one row per run,
with the held-out money of the trained policy and the `net_position` baseline.

Everything else (learning curves, VTI against the ceiling per test period,
participants, trades, the grid) is on the Studio's **Results** page: start it with
`python run_studio.py` and pick the experiment there.

In [ ]:
from core.runner import results

if NAME:
    refs = results.find_runs(experiments=[(TARGET, NAME)])
    found = results.load_runs(refs)
    display(results.overview(found))